# EDA — ChurnGuard (datos reales)

**Qué es:** análisis exploratorio sobre el dataset real `data/playnova_real.db` (16.447 jugadores etiquetados de 18.099 reseñas públicas de Steam, 6 F2P, sep-2026).

**De dónde sale todo:** cada número y figura de este notebook sale de la base de datos. Las figuras se guardan en `docs/img/eda_*.png` y alimentan la web (`web/`) y `docs/04_analisis.md`.

**Cómo usarlo:** abrir en Jupyter (`jupyter notebook notebooks/EDA.ipynb`) y ejecutar de arriba a abajo. No necesita Streamlit.

> Nota honesta: las tasas por título son tasas *dentro de la muestra de reseñistas*, no la retención global de esos juegos. Lo que generaliza son las relaciones (validadas juego por juego en `src/eval_per_title.py`).

In [ ]:
import json
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("dark_background")

CYAN, VIOLET, RED, AMBER = "#22d3ee", "#8b5cf6", "#ef4444", "#f59e0b"
NAMES = {570: "Dota 2", 440: "TF2", 230410: "Warframe", 238960: "PoE",
         1172470: "Apex", 1097150: "Fall Guys"}
ORDER = ["Dota 2", "Apex", "TF2", "Warframe", "PoE", "Fall Guys"]

# Rutas robustas: funciona ejecutado desde la raíz o desde notebooks/
DB_CANDIDATES = ["data/playnova_real.db", "../data/playnova_real.db"]
IMG_CANDIDATES = ["docs/img", "../docs/img"]
METRICS_CANDIDATES = ["models/real_metrics.json", "../models/real_metrics.json"]

DB = next(p for p in DB_CANDIDATES if os.path.exists(p))
IMG = next(p for p in IMG_CANDIDATES if os.path.exists(p))
METRICS = next(p for p in METRICS_CANDIDATES if os.path.exists(p))
print("DB:", DB, "| IMG:", IMG, "| METRICS:", METRICS)


def style_ax(ax, title, xlabel="", ylabel=""):
    ax.set_title(title, color=CYAN, fontsize=12, pad=10)
    ax.set_xlabel(xlabel, color="#a0a0c0")
    ax.set_ylabel(ylabel, color="#a0a0c0")
    ax.tick_params(colors="#a0a0c0")

## 0 · Carga y resumen global

In [ ]:
conn = sqlite3.connect(DB)
df = pd.read_sql("SELECT * FROM reviews WHERE churn IS NOT NULL", conn)
conn.close()
df["title"] = df["appid"].map(NAMES)

print(f"Filas: {len(df)} · churn={df['churn'].mean():.1%} · "
      f"high_value={df['high_value'].mean():.1%} · recomienda={df['voted_up'].mean():.1%}")
df.groupby("title").agg(n=("churn", "size"), churn=("churn", "mean"),
    high_value=("high_value", "mean"),
    med_horas_reseña=("hours_at_review", "median")).round({"churn": 3, "high_value": 3, "med_horas_reseña": 1})

## 1 · Estado del jugador por título

Juegos vivos vs juegos fríos: dónde priorizar Live-Ops. Churn (inactivos hoy) vs high-value (proxy top-25% horas + recomienda).

In [ ]:
t = df.groupby("title").agg(churn=("churn", "mean"), high_value=("high_value", "mean")).loc[ORDER]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(t))
ax.bar(x - 0.2, t["churn"] * 100, 0.4, label="Churn (inactivos hoy)", color=RED)
ax.bar(x + 0.2, t["high_value"] * 100, 0.4, label="High-value (proxy)", color=CYAN)
ax.set_xticks(x, t.index)
style_ax(ax, "1 · Estado del jugador por título (muestra de reseñistas, sep-2026)", ylabel="% de reseñistas")
leg = ax.legend()
leg.get_frame().set_facecolor("#1e1b3a")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_1_estado_titulo.png", bbox_inches="tight")
plt.show()

## 2 · El engagement temprano decide

Churn por cuartil de horas jugadas en el momento de la reseña. Q1 (pocas horas) vs Q4 (muchas horas).

In [ ]:
d = df.copy()
d["q"] = pd.qcut(d["hours_at_review"], 4, labels=["Q1\n(pocas h.)", "Q2", "Q3", "Q4\n(muchas h.)"])
t2 = d.groupby("q", observed=True)["churn"].mean()

fig, ax = plt.subplots(figsize=(8, 5))
t2.plot(kind="bar", ax=ax, color=VIOLET, edgecolor=CYAN)
for i, v in enumerate(t2.values):
    ax.text(i, v + 0.008, f"{v:.1%}", ha="center", color="#e8e8f0", fontsize=10)
style_ax(ax, "2 · Churn según horas jugadas en el momento de la reseña",
         xlabel="Cuartil de engagement temprano", ylabel="Tasa de churn")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_2_engagement.png", bbox_inches="tight")
plt.show()
t2

## 3 · Satisfacción ≠ retención

Recomendar el juego apenas mueve la aguja del churn. Gustar no retiene; retiene el hábito temprano.

In [ ]:
t3 = df.groupby("voted_up")["churn"].mean()

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(["No recomienda", "Recomienda"], t3.values * 100, color=[AMBER, CYAN])
for i, v in enumerate(t3.values * 100):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", color="#e8e8f0")
style_ax(ax, "3 · Satisfacción ≠ retención", ylabel="Tasa de churn")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_3_voto.png", bbox_inches="tight")
plt.show()
t3

## 4 · Las primeras horas separan (distribución)

Distribución de horas tempranas (escala log1p): churned vs activos, con sus medianas.

In [ ]:
med_ch = df.loc[df["churn"] == 1, "hours_at_review"].median()
med_ac = df.loc[df["churn"] == 0, "hours_at_review"].median()
print(f"Mediana horas en reseña: churned {med_ch:.1f} h vs activos {med_ac:.1f} h")

fig, ax = plt.subplots(figsize=(8, 5))
ch = np.log1p(df.loc[df["churn"] == 1, "hours_at_review"])
ac = np.log1p(df.loc[df["churn"] == 0, "hours_at_review"])
ax.hist(ch, bins=40, alpha=0.6, label="Churned", color=RED)
ax.hist(ac, bins=40, alpha=0.5, label="Activos", color=CYAN)
ax.axvline(np.log1p(med_ch), color=RED, linestyle="--", label="Mediana churned")
ax.axvline(np.log1p(med_ac), color=CYAN, linestyle="--", label="Mediana activos")
style_ax(ax, "4 · Las primeras horas separan: distribución (log1p)",
         xlabel="log(1 + horas en reseña)", ylabel="N.º jugadores")
leg = ax.legend()
leg.get_frame().set_facecolor("#1e1b3a")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_4_dist_horas.png", bbox_inches="tight")
plt.show()

## 5 · La antigüedad de la señal pesa

Churn por cuartil de `review_age_days`. Es ritmo de reseñas = fase del título (limitación documentada: ventana desigual entre juegos).

In [ ]:
d = df.copy()
d["q_age"] = pd.qcut(d["review_age_days"], 4, labels=["Reciente", "Q2", "Q3", "Antigua"])
t5 = d.groupby("q_age", observed=True)["churn"].mean()

fig, ax = plt.subplots(figsize=(8, 5))
t5.plot(kind="bar", ax=ax, color=AMBER, edgecolor=CYAN)
for i, v in enumerate(t5.values):
    ax.text(i, v + 0.008, f"{v:.1%}", ha="center", color="#e8e8f0", fontsize=10)
style_ax(ax, "5 · Churn vs antigüedad de la señal (review_age_days)",
         xlabel="Cuartil de antigüedad", ylabel="Tasa de churn")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_5_antiguedad.png", bbox_inches="tight")
plt.show()
t5

## 6 · High-value estable por título

Proxy transparente (top-25% horas + recomienda): estable entre 15–24% por título.

In [ ]:
t6 = df.groupby("title").agg(hv=("high_value", "mean")).loc[ORDER]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(t6.index, t6["hv"] * 100, color=CYAN, edgecolor=VIOLET)
ax.axhline(df["high_value"].mean() * 100, color=AMBER, linestyle="--",
           label=f"Global {df['high_value'].mean():.1%}")
for i, v in enumerate(t6["hv"].values * 100):
    ax.text(i, v + 0.4, f"{v:.1f}%", ha="center", color="#e8e8f0", fontsize=9)
style_ax(ax, "6 · High-value estable por título (proxy top-25% horas + recomienda)",
         ylabel="% high-value")
leg = ax.legend()
leg.get_frame().set_facecolor("#1e1b3a")
ax.tick_params(axis="x", rotation=12)
plt.tight_layout()
plt.savefig(f"{IMG}/eda_6_highvalue.png", bbox_inches="tight")
plt.show()
t6

## 7 · Qué pesa en los modelos

Importancias reales de los 2 modelos (`src/train_real.py`, `models/real_metrics.json`).

In [ ]:
with open(METRICS, encoding="utf-8") as f:
    m = json.load(f)
ch = pd.Series(m["churn_model"]["top_features"]).sort_values()
cv = pd.Series(m["conversion_model"]["top_features"]).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ch.plot(kind="barh", ax=axes[0], color=VIOLET, edgecolor=CYAN)
axes[0].set_title("7a · Churn: qué pesa (importancias RF)", color=CYAN, fontsize=11)
cv.plot(kind="barh", ax=axes[1], color=CYAN, edgecolor=VIOLET)
axes[1].set_title("7b · High-value: qué pesa (importancias RF)", color=CYAN, fontsize=11)
for a in axes:
    a.tick_params(colors="#a0a0c0")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_7_importancias.png", bbox_inches="tight")
plt.show()
print("churn ROC:", m["churn_model"].get("roc_auc"), "| F1:", m["churn_model"].get("f1"))
print("conversion ROC:", m["conversion_model"].get("roc_auc"), "| F1:", m["conversion_model"].get("f1"))

## 8 · Cómo se construye la etiqueta churn (sanity check, NO es feature)

**Regla** (`src/build_real_dataset.py`): churn=1 si `hours_l2w==0` + `hours_forever>=2h` + `days_since_last_played>30d`; churn=0 si `hours_l2w>0`; resto = ambiguo (NULL, 1.652 casos excluidos del modelo).

**Lectura:** la figura 8a muestra un escalón perfecto en 30 días y la 8b que `hours_l2w==0` equivale a churned. Es *por construcción*, no un descubrimiento. Por eso `hours_l2w`, `days_since_last_played` y `hours_forever` **nunca entran al modelo** (`src/train_real.py`): usarlas sería fuga de etiqueta.

**Implicación:** el modelo solo usa observables del momento de la reseña + cuenta. Si en producción tienes recencia real, úsala como *regla de negocio*, no como feature del mismo modelo.

In [ ]:
import sqlite3
con0 = sqlite3.connect(DB)
amb = pd.read_sql("SELECT churn, COUNT(*) n FROM reviews GROUP BY churn", con0)
con0.close()
print(amb.to_string(index=False))
print(f"Ambiguos excluidos: {int(amb.loc[amb['churn'].isna(), 'n'].sum())} "
      f"({amb.loc[amb['churn'].isna(), 'n'].sum() / amb['n'].sum():.1%} del crudo)")
print(pd.crosstab(df["hours_l2w"] == 0, df["churn"]).to_string())

bins8 = pd.cut(df["days_since_last_played"], [0, 7, 30, 180, 1e9],
               labels=["0-7d", "8-30d", "31-180d", "181d+"])
t8 = df.groupby(bins8, observed=True)["churn"].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
t8.plot(kind="bar", ax=axes[0], color=[CYAN, CYAN, RED, RED], edgecolor="white")
for i, v in enumerate(t8.values):
    axes[0].text(i, v + 0.02, f"{v:.1%}", ha="center", color="#e8e8f0", fontsize=10)
style_ax(axes[0], "8a · Churn por recencia (define la etiqueta: corte en 30d)",
         xlabel="Días desde última partida", ylabel="Tasa de churn")
h = np.log1p(df["hours_l2w"])
axes[1].hist(h[df["hours_l2w"] == 0], bins=5, alpha=0.7,
             label="0 h (27%: candidatos a churn)", color=RED)
axes[1].hist(h[df["hours_l2w"] > 0], bins=40, alpha=0.5,
             label=">0 h (activos por definición)", color=CYAN)
style_ax(axes[1], "8b · Horas últimas 2 sem. (log1p): separan por construcción",
         xlabel="log(1 + horas 2 sem)", ylabel="N.º jugadores")
leg = axes[1].legend()
leg.get_frame().set_facecolor("#1e1b3a")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_8_etiqueta.png", bbox_inches="tight")
plt.show()
t8

## 9 · Correlaciones: casi nada ata fuerte (salvo lo que define la etiqueta)

Matriz de Pearson entre numéricas. Solo `review_age_days` (+0,63) y `days_since_last_played` (+0,51) correlacionan fuerte con churn — y ambas están contaminadas: la primera por la **ventana de muestreo desigual** (sección 12) y la segunda **por construcción** (sección 8).

**Lectura:** `hours_at_review` (-0,13), `voted_up` (-0,03), `review_len` (+0,06), `num_games_owned` (+0,09) apenas correlacionan en lineal. El churn es multifactorial y no lineal: por eso el Random Forest (ROC 0,93) aporta sobre cualquier regla simple.

**Limitación honesta:** Pearson no captura relaciones no lineales ni interacciones (ver secciones 10 y 13, donde horas × voto sí separan).

In [ ]:
cols9 = ["churn", "voted_up", "hours_at_review", "hours_forever", "hours_l2w",
         "days_since_last_played", "review_age_days", "review_len",
         "num_games_owned", "num_reviews"]
c9 = df[cols9].corr(numeric_only=True)
print(c9["churn"].sort_values().to_string())

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(c9.values, vmin=-0.7, vmax=0.7, cmap="coolwarm")
ax.set_xticks(range(len(cols9)), cols9, rotation=35, ha="right", fontsize=8, color="#a0a0c0")
ax.set_yticks(range(len(cols9)), cols9, fontsize=8, color="#a0a0c0")
for i in range(len(cols9)):
    for j in range(len(cols9)):
        ax.text(j, i, f"{c9.values[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(c9.values[i, j]) > 0.35 else "#a0a0c0")
style_ax(ax, "9 · Correlación entre numéricas (Pearson): solo recencia/antigüedad atan fuerte")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig(f"{IMG}/eda_9_corr.png", bbox_inches="tight")
plt.show()

## 10 · El engagement protege en los 6 juegos (no es un artefacto de Fall Guys)

Corte por la mediana global de horas en reseña (80,6 h): churn con horas bajas vs altas, **dentro de cada título**.

**Lectura:** el orden se mantiene en los 6 (Dota 8%→4%, Apex 6%→4%, TF2 9%→8%, Warframe 10%→8%, PoE 59%→45%, Fall Guys 94%→81%). La magnitud cambia —en juegos fríos hasta los enganchados se van mucho— pero la dirección es universal.

**Implicación:** es la evidencia EDA de que la relación generaliza, antes de la prueba formal juego por juego (`src/eval_per_title.py`: churn ROC 0,891–0,924 en los 6).

In [ ]:
med10 = df["hours_at_review"].median()
print(f"Mediana global horas en reseña: {med10:.1f} h")
rows10 = []
for t in ORDER:
    g = df[df["title"] == t]
    lo = g.loc[g["hours_at_review"] <= med10, "churn"].mean()
    hi = g.loc[g["hours_at_review"] > med10, "churn"].mean()
    rows10.append((t, lo, hi, len(g)))
t10 = pd.DataFrame(rows10, columns=["title", "bajas", "altas", "n"]).set_index("title")
print(t10.round(3).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(t10))
ax.bar(x - 0.2, t10["bajas"] * 100, 0.4, label=f"Horas bajas (≤ {med10:.0f} h)", color=RED)
ax.bar(x + 0.2, t10["altas"] * 100, 0.4, label="Horas altas", color=CYAN)
for i in range(len(t10)):
    ax.text(i - 0.2, t10["bajas"].iloc[i] * 100 + 1, f"{t10['bajas'].iloc[i]:.0%}",
            ha="center", fontsize=8, color="#e8e8f0")
    ax.text(i + 0.2, t10["altas"].iloc[i] * 100 + 1, f"{t10['altas'].iloc[i]:.0%}",
            ha="center", fontsize=8, color="#e8e8f0")
ax.set_xticks(x, t10.index)
ax.tick_params(axis="x", rotation=12)
style_ax(ax, "10 · El engagement protege en los 6 juegos (corte = mediana global)",
         ylabel="% churn")
leg = ax.legend()
leg.get_frame().set_facecolor("#1e1b3a")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_10_engagement_titulo.png", bbox_inches="tight")
plt.show()
t10

## 11 · Quién escribe: idioma, biblioteca y esfuerzo de la reseña

**Lectura:**
- *Idioma (top-8):* ruso 12,9% y chino simplificado 18,0% abajo; brasileño 49,0% y francés 46,6% arriba. **No es causal**: el idioma correlaciona con el título que cada comunidad juega.
- *Biblioteca:* forma de U — 0 juegos (52,6% de filas, Steam oculta el dato) 25,5% · 1–20 juegos 13,3% (el mínimo) · 21–100 28,7% · 101+ 47,8%. Tener biblioteca enorme no protege; el perfil coleccionista prueba y abandona.
- *Longitud de reseña:* casi plana (Q1 25,3% → Q4 30,9%). Escribir mucho no predice quedarse; en el modelo pesa poco (0,036).

**Limitación:** `num_games_owned=0` no significa cuenta vacía, sino dato no visible. Se modela con `log1p` y el bosque lo trata como una rama más, no como ausencia real.

In [ ]:
top8 = df["language"].value_counts().head(8).index.tolist()
t11a = df[df["language"].isin(top8)].groupby("language")["churn"].mean().sort_values()
print(t11a.round(3).to_string())
g11 = pd.cut(df["num_games_owned"], [-1, 0, 20, 100, 1e9],
             labels=["0 (52%: oculto)", "1-20", "21-100", "101+"])
t11b = df.groupby(g11, observed=True)["churn"].mean()
print(t11b.round(3).to_string())
df["rl_q"] = pd.qcut(df["review_len"], 4, labels=["Q1 cortas", "Q2", "Q3", "Q4 largas"])
t11c = df.groupby("rl_q", observed=True)["churn"].mean()
print(t11c.round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
t11a.plot(kind="barh", ax=axes[0], color=VIOLET, edgecolor=CYAN)
style_ax(axes[0], "11a · Churn por idioma (top-8)", xlabel="Tasa de churn")
t11b.plot(kind="bar", ax=axes[1], color=AMBER, edgecolor=CYAN)
for i, v in enumerate(t11b.values):
    axes[1].text(i, v + 0.008, f"{v:.0%}", ha="center", fontsize=8, color="#e8e8f0")
style_ax(axes[1], "11b · Churn por biblioteca Steam", xlabel="Juegos en propiedad",
         ylabel="Tasa de churn")
axes[1].tick_params(axis="x", rotation=15)
t11c.plot(kind="bar", ax=axes[2], color=CYAN, edgecolor=VIOLET)
for i, v in enumerate(t11c.values):
    axes[2].text(i, v + 0.008, f"{v:.0%}", ha="center", fontsize=8, color="#e8e8f0")
style_ax(axes[2], "11c · Churn por longitud de reseña", xlabel="Cuartil de longitud",
         ylabel="Tasa de churn")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_11_perfil.png", bbox_inches="tight")
plt.show()

## 12 · Ventana de muestreo desigual: la limitación más importante

Cada título se muestreó en una ventana temporal distinta: Dota 2 concentra ~3.000 reseñas en **3,4 días**; Fall Guys necesita **469 días** para las mismas 3.000. La mediana de antigüedad va de 1,9 días (Dota) a 269,7 días (Fall Guys).

**Lectura:** una reseña antigua = más tiempo para haberse ido + fase distinta del juego. Por eso `review_age_days` domina la importancia del modelo de churn (0,50): actúa como **proxy de título/fase**, no como causa.

**Cómo lo compensamos:** validación juego por juego (`src/eval_per_title.py`) — el modelo discrimina jugadores *dentro* de cada comunidad (ROC 0,891–0,924), así que no se limita a memorizar 'Fall Guys = churn'. Y las tasas por título se reportan siempre como *tasas en muestra de reseñistas*, nunca como retención global.

In [ ]:
t12 = df.groupby("title")["review_age_days"].agg(["median", "min", "max", "count"])
span12 = df.groupby("title")["timestamp_created"].agg(lambda s: (s.max() - s.min()) / 86400)
t12["span_dias"] = span12
print(t12.loc[ORDER].round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
vals12 = [df.loc[df["title"] == t, "review_age_days"].values for t in ORDER]
axes[0].boxplot(vals12, showfliers=False)
axes[0].set_xticklabels(ORDER, rotation=15, color="#a0a0c0")
style_ax(axes[0], "12a · Antigüedad de la reseña por título (caja sin atípicos)",
         ylabel="Días de antigüedad")
span12.loc[ORDER].plot(kind="bar", ax=axes[1], color=AMBER, edgecolor=CYAN)
for i, v in enumerate(span12.loc[ORDER].values):
    axes[1].text(i, v + 8, f"{v:.0f} d", ha="center", fontsize=9, color="#e8e8f0")
style_ax(axes[1], "12b · Ventana de muestreo por título (max-min fecha reseña)",
         ylabel="Días de ventana")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(f"{IMG}/eda_12_ventana.png", bbox_inches="tight")
plt.show()
t12

## 13 · Voto × hábito: el hábito manda, el voto solo matiza a los tibios

Tabla 2×2 (mediana global 80,6 h): horas bajas + no recomienda **43,1%** de churn vs horas altas + recomienda **18,2%**.

**Lectura fina:** con horas altas el voto es irrelevante (17,8% vs 18,2%). Con horas bajas, no recomendar suma ~8 pp (43,1% vs 34,5%). Traducción a producto: al jugador enganchado no lo mueve la nota; al tibio, el enfado sí lo empuja fuera.

**Implicación:** la prioridad Live-Ops es (1) horas tempranas, (2) voto solo como desempate en la banda baja.

In [ ]:
med13 = df["hours_at_review"].median()
df["band13"] = np.where(df["hours_at_review"] <= med13, "Horas bajas", "Horas altas")
t13 = df.groupby(["voted_up", "band13"])["churn"].mean().unstack()
print(t13.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
labels13 = ["No recomienda", "Recomienda"]
x = np.arange(2)
lo13 = [t13.loc[0, "Horas bajas"], t13.loc[1, "Horas bajas"]]
hi13 = [t13.loc[0, "Horas altas"], t13.loc[1, "Horas altas"]]
ax.bar(x - 0.2, np.array(lo13) * 100, 0.4, label="Horas bajas", color=RED)
ax.bar(x + 0.2, np.array(hi13) * 100, 0.4, label="Horas altas", color=CYAN)
for i in range(2):
    ax.text(i - 0.2, lo13[i] * 100 + 0.7, f"{lo13[i]:.1%}", ha="center", color="#e8e8f0")
    ax.text(i + 0.2, hi13[i] * 100 + 0.7, f"{hi13[i]:.1%}", ha="center", color="#e8e8f0")
ax.set_xticks(x, labels13)
style_ax(ax, "13 · Voto x hábito: el hábito separa ~20 pp; el voto solo ~3 pp",
         ylabel="Tasa de churn")
leg = ax.legend()
leg.get_frame().set_facecolor("#1e1b3a")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_13_voto_habito.png", bbox_inches="tight")
plt.show()
t13

## 14 · Calidad de datos: ceros, outliers y constantes (qué se hizo)

- **Ceros:** `num_games_owned` 52,6% en 0 (Steam oculta el dato, no cuenta vacía) · `hours_l2w` 27,1% en 0 (por construcción = churned) · resto <8%.
- **Outliers reales:** `num_games_owned` max 32.502 · `num_reviews` max 26.464 · `review_len` max 7.985 caracteres. No se recortan: el bosque + `log1p` los absorben.
- **Constantes:** `early_access` y `refunded` son 0 en el 100% (importancia 0 en ambos modelos — ruido documentado, no señal).
- **Muestra tiny:** `received_free` solo 81 casos (0,5%) con churn 93,8%: se reporta pero no se decide nada con n=81.
- **Sin nulos** en las 16.447 filas etiquetadas; los 1.652 ambiguos se excluyen y se cuentan (sección 8).

**Tratamiento:** `log1p` en horas/longitud/juegos (`src/train_real.py`), idioma con LabelEncoder + cola a 'other', y RandomForest con `class_weight=balanced_subsample`.

In [ ]:
numcols14 = ["hours_at_review", "hours_forever", "hours_l2w", "days_since_last_played",
              "review_age_days", "review_len", "num_games_owned", "num_reviews"]
z14 = pd.Series({c: (df[c] == 0).mean() * 100 for c in numcols14}).sort_values(ascending=False)
print(z14.round(1).to_string())
print("early_access únicos:", df["early_access"].unique(), "| refunded únicos:",
      df["refunded"].unique())
print(f"received_free: n={int(df['received_free'].sum())} "
      f"churn|free={df.loc[df['received_free'] == 1, 'churn'].mean():.1%}")
print(f"max juegos={df['num_games_owned'].max():.0f} | max reviews={df['num_reviews'].max():.0f} "
      f"| max len={df['review_len'].max():.0f} | nulos={int(df.isna().sum().sum())}")

fig, ax = plt.subplots(figsize=(9, 5))
z14.plot(kind="barh", ax=ax, color=RED, edgecolor=CYAN)
for i, v in enumerate(z14.values):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=9, color="#e8e8f0")
style_ax(ax, "14 · Ceros por columna numérica (% en 0): ojo biblioteca y últimas 2 sem.",
         xlabel="% en cero")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_14_calidad.png", bbox_inches="tight")
plt.show()
z14

## Descubrimientos (números de este EDA)

- **D1 · Engagement temprano decide:** Q1 → Q4 cae del ~35% al ~13%. Mediana churned ~37 h vs activos ~121 h.
- **D2 · Cada título vive su fase:** Dota 2 / Apex (vivos) vs PoE / Fall Guys (fríos). Tasas en muestra de reseñistas, no retención global.
- **D3 · Satisfacción ≠ retención:** recomienda ~26,5% vs no recomienda ~29,6%. Gustar no retiene; retiene el hábito.
- **D4 · High-value estable:** ~20% global, 15–24% por título (proxy top-25% horas + recomienda).
- **D5 · Antigüedad pesa:** es ritmo de reseñas = fase del título (ventana desigual, documentado como limitación).
- **D6 · Modelos:** churn ROC 0.930 / F1 0.817; conversión ROC 0.972 / F1 0.847; validez por título churn ROC 0.891–0.924 (`src/eval_per_title.py`).
- **D7 · La etiqueta se verifica (8):** el escalón en 30 días y `hours_l2w==0` ⇔ churn es por construcción; esas columnas jamás entran al modelo (anti-fuga).
- **D8 · Nada lineal ata salvo recencia/antigüedad (9):** resto de correlaciones |r|<0,15 → el churn es multifactorial; el bosque aporta.
- **D9 · El engagement vale en los 6 juegos (10):** horas bajas vs altas separan en todos (Dota 8%→4% … Fall Guys 94%→81%).
- **D10 · Perfil del reseñista (11):** biblioteca en U (mínimo 13,3% en 1–20 juegos), idioma no causal, longitud de reseña plana.
- **D11 · Ventana desigual (12):** Dota 3,4 días vs Fall Guys 469 días de muestreo — `review_age_days` es proxy de fase, compensado con validación por título.
- **D12 · Voto × hábito (13):** con horas altas el voto no mueve (17,8% vs 18,2%); con horas bajas suma ~8 pp (43,1% vs 34,5%).
- **D13 · Calidad (14):** 52,6% ceros en biblioteca (dato oculto), `early_access`/`refunded` constantes, outliers absorbidos con `log1p`.

Figuras guardadas en `docs/img/eda_*.png`. Detalle narrado en `docs/04_analisis.md`.